In [ ]:
pip install yfinance pandas numpy matplotlib openpyxl

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import datetime as dt
import warnings
import os
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
#  GOOFY SCREENER v2 — Phase 3
#  Upgrades over v1:
#    - Grid search per strategy (finds best params per asset, not fixed defaults)
#    - Stock universe auto-loader (ASX top stocks / US top stocks / custom)
#    - Excel output with colour-coded results saved daily
#    - Runs as a daily automated script
# ══════════════════════════════════════════════════════════════════════════════

# ── CONFIG — change these to control what the screener does ───────────────────

# Choose universe: "ASX", "US", or "CUSTOM"
UNIVERSE = "ASX"

# If UNIVERSE = "CUSTOM", put your tickers here
CUSTOM_ASSETS = ["NVDA", "SPY", "CBA.AX", "BHP.AX", "7203.T", "6758.T"]

# Minimum rows of data required — assets with less are skipped
MIN_ROWS = 500

# Train/test split (same as all individual notebooks)
TRAIN_START = dt.datetime(2016, 1, 1)
TRAIN_END   = dt.datetime(2021, 1, 1)
TEST_END    = dt.datetime.now()

PERIODS_PER_YEAR = 252

# Output folder for Excel + charts
OUTPUT_DIR = "screener_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Stock universes ────────────────────────────────────────────────────────────
ASX_UNIVERSE = [
    # Big 4 Banks
    "CBA.AX", "WBC.AX", "ANZ.AX", "NAB.AX",
    # Mining / Resources
    "BHP.AX", "RIO.AX", "FMG.AX", "S32.AX", "MIN.AX",
    # Energy
    "WDS.AX", "STO.AX", "BPT.AX",
    # Healthcare
    "CSL.AX", "RMD.AX", "COH.AX", "SHL.AX",
    # Retail / Consumer
    "WES.AX", "WOW.AX", "COL.AX",
    # Tech / Fintech
    "XRO.AX", "CPU.AX", "WTC.AX",
    # Infrastructure / Utilities
    "TCL.AX", "APA.AX", "AGL.AX",
    # ETFs
    "IOZ.AX", "STW.AX", "VAS.AX",
]

US_UNIVERSE = [
    # Tech
    "AAPL", "MSFT", "NVDA", "GOOGL", "META", "AMZN", "TSLA",
    # Finance
    "JPM", "BAC", "GS", "V", "MA",
    # Healthcare
    "JNJ", "UNH", "PFE", "ABBV",
    # Energy
    "XOM", "CVX",
    # Consumer
    "PG", "KO", "MCD", "WMT",
    # ETFs
    "SPY", "QQQ", "IWM", "GLD",
]

# Select universe
if UNIVERSE == "ASX":
    ASSETS = ASX_UNIVERSE
elif UNIVERSE == "US":
    ASSETS = US_UNIVERSE
else:
    ASSETS = CUSTOM_ASSETS

print(f"GOOFY SCREENER v2")
print(f"Universe: {UNIVERSE} ({len(ASSETS)} assets)")
print(f"Train: {TRAIN_START.date()} → {TRAIN_END.date()}")
print(f"Test:  {TRAIN_END.date()} → {TEST_END.date()}")
print(f"Output: {OUTPUT_DIR}/")
print("\nDownloading price data...")

price_data = {}
skipped    = []
for asset in ASSETS:
    try:
        raw = yf.download(asset, start=TRAIN_START, end=TEST_END,
                          auto_adjust=True, progress=False)
        if raw.empty or len(raw) < MIN_ROWS:
            skipped.append(asset)
            print(f"  SKIP {asset}: only {len(raw)} rows (need {MIN_ROWS})")
        else:
            price_data[asset] = raw["Close"].squeeze()
            print(f"  OK   {asset}: {len(raw)} rows")
    except Exception as e:
        skipped.append(asset)
        print(f"  ERROR {asset}: {e}")

ASSETS = list(price_data.keys())
print(f"\nReady: {len(ASSETS)} assets | Skipped: {len(skipped)}")
if skipped:
    print(f"Skipped: {skipped}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STRATEGY FUNCTIONS + GRID SEARCH PARAMS
#  Each strategy now has a param grid — the screener tries all combos on
#  training data and picks the best set per asset before testing
# ══════════════════════════════════════════════════════════════════════════════

# ── Shared metrics ─────────────────────────────────────────────────────────────
def compute_metrics(price_series, position_series):
    df = pd.DataFrame({"price": price_series})
    df["position"]     = position_series.reindex(df.index).fillna(0)
    df["log_ret"]      = np.log(df["price"] / df["price"].shift(1))
    df["strat_ret"]    = df["position"] * df["log_ret"]
    df["cum_market"]   = np.exp(df["log_ret"].cumsum())
    df["cum_strategy"] = np.exp(df["strat_ret"].cumsum())

    strat_lr  = df["strat_ret"].dropna()
    if len(strat_lr) < 10:
        return {"Sharpe": np.nan, "Total Ret %": np.nan,
                "Ann Ret %": np.nan, "Max DD %": np.nan, "_df": df}

    ann_ret   = np.exp(strat_lr.mean() * PERIODS_PER_YEAR) - 1
    vol       = strat_lr.std() * np.sqrt(PERIODS_PER_YEAR)
    sharpe    = ann_ret / vol if vol != 0 else np.nan
    cum       = df["cum_strategy"].dropna()
    total_ret = cum.iloc[-1] - 1 if len(cum) > 0 else np.nan
    mdd       = ((cum - cum.cummax()) / cum.cummax()).min() if len(cum) > 0 else np.nan

    return {
        "Sharpe":      round(float(sharpe),    3),
        "Total Ret %": round(float(total_ret) * 100, 2),
        "Ann Ret %":   round(float(ann_ret)   * 100, 2),
        "Max DD %":    round(float(mdd)        * 100, 2),
        "_df":         df,
    }


# ── Strategy functions ─────────────────────────────────────────────────────────
def strategy_ma(price, short=20, long=50):
    sig = (price.rolling(short).mean() > price.rolling(long).mean()).astype(int)
    return sig.shift(1)

def strategy_rsi(price, period=14, oversold=30, overbought=70):
    delta = price.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rsi   = 100 - (100 / (1 + gain / loss))
    sig   = np.zeros(len(price))
    hold  = False
    for i in range(len(rsi)):
        r = rsi.iloc[i]
        if np.isnan(r):       sig[i] = 0
        elif not hold and r < oversold:  hold = True;  sig[i] = 1
        elif hold and r > overbought:    hold = False; sig[i] = 0
        else:                 sig[i] = 1 if hold else 0
    return pd.Series(sig, index=price.index).shift(1)

def strategy_bb(price, window=20, num_std=2.0):
    mid  = price.rolling(window).mean()
    low  = mid - num_std * price.rolling(window).std()
    sig  = np.zeros(len(price))
    hold = False
    for i in range(len(price)):
        p, m, l = price.iloc[i], mid.iloc[i], low.iloc[i]
        if np.isnan(l):          sig[i] = 0
        elif not hold and p <= l: hold = True;  sig[i] = 1
        elif hold and p >= m:     hold = False; sig[i] = 0
        else:                    sig[i] = 1 if hold else 0
    return pd.Series(sig, index=price.index).shift(1)

def strategy_macd(price, fast=12, slow=26, signal_p=9):
    macd = price.ewm(span=fast, adjust=False).mean() - price.ewm(span=slow, adjust=False).mean()
    sig  = macd.ewm(span=signal_p, adjust=False).mean()
    return (macd > sig).astype(int).shift(1)

def strategy_mr(price, window=20, threshold=1.5):
    roll_mean = price.rolling(window).mean()
    z         = (price - roll_mean) / price.rolling(window).std()
    sig       = np.zeros(len(price))
    hold      = False
    for i in range(len(z)):
        zi = z.iloc[i]
        if np.isnan(zi):              sig[i] = 0
        elif not hold and zi < -threshold: hold = True;  sig[i] = 1
        elif hold and zi >= 0:             hold = False; sig[i] = 0
        else:                         sig[i] = 1 if hold else 0
    return pd.Series(sig, index=price.index).shift(1)


# ── Param grids per strategy ───────────────────────────────────────────────────
# Smaller than individual notebooks (for speed across many assets)
# but still meaningful enough to find asset-specific best params
STRATEGY_GRIDS = {
    "MA Crossover": [
        {"short": s, "long": l}
        for s in [10, 20, 50]
        for l in [50, 100, 200]
        if s < l
    ],
    "RSI": [
        {"period": p, "oversold": os, "overbought": ob}
        for p in [10, 14, 21]
        for os, ob in [(25, 65), (30, 70), (35, 75)]
    ],
    "Bollinger Bands": [
        {"window": w, "num_std": s}
        for w in [10, 20, 30]
        for s in [1.5, 2.0, 2.5]
    ],
    "MACD": [
        {"fast": f, "slow": s, "signal_p": sig}
        for f in [8, 12]
        for s in [21, 26]
        for sig in [7, 9]
        if f < s
    ],
    "Mean Reversion": [
        {"window": w, "threshold": t}
        for w in [10, 20, 40]
        for t in [1.0, 1.5, 2.0]
    ],
}

STRATEGY_FNS = {
    "MA Crossover":   strategy_ma,
    "RSI":            strategy_rsi,
    "Bollinger Bands": strategy_bb,
    "MACD":           strategy_macd,
    "Mean Reversion": strategy_mr,
}

STRATEGY_COLORS = {
    "MA Crossover":    "steelblue",
    "RSI":             "darkorange",
    "Bollinger Bands": "green",
    "MACD":            "purple",
    "Mean Reversion":  "crimson",
}

total_combos = sum(len(v) for v in STRATEGY_GRIDS.values())
print(f"Strategy grid loaded: {total_combos} param combos per asset")
for name, grid in STRATEGY_GRIDS.items():
    print(f"  {name}: {len(grid)} combos")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SCREENER ENGINE — grid search + robustness check
#  For each asset:
#    1. Try all param combos for all 5 strategies on TRAIN data
#    2. Pick the strategy + params with the highest train Sharpe
#    3. ROBUSTNESS CHECK: how many other param combos for that strategy
#       also had positive Sharpe? High = genuinely robust. Low = possibly lucky.
#    4. Apply best config to TEST data (unseen)
#    5. Compare strategy Max DD vs B&H Max DD (DD Saved = protection provided)
# ══════════════════════════════════════════════════════════════════════════════

screener_results = []
test_dfs         = {}
best_configs     = {}

print(f"Running screener on {len(ASSETS)} assets...")
print("=" * 80)

for idx, asset in enumerate(ASSETS):
    print(f"[{idx+1}/{len(ASSETS)}] {asset}", end=" ... ")

    full_price  = price_data[asset]
    train_price = full_price[full_price.index < TRAIN_END]
    test_price  = full_price[full_price.index >= TRAIN_END]

    if len(train_price) < 100 or len(test_price) < 50:
        print("SKIP (insufficient data)")
        continue

    # ── Grid search on training data ───────────────────────────────────────────
    best_sharpe = -999
    best_strat  = None
    best_params = None

    for strat_name, fn in STRATEGY_FNS.items():
        for params in STRATEGY_GRIDS[strat_name]:
            try:
                pos     = fn(train_price, **params)
                metrics = compute_metrics(train_price, pos)
                s       = metrics["Sharpe"]
                if not np.isnan(s) and s > best_sharpe:
                    best_sharpe = s
                    best_strat  = strat_name
                    best_params = params
            except:
                continue

    if best_strat is None:
        print("SKIP (no valid strategy found)")
        continue

    # ── Robustness check ───────────────────────────────────────────────────────
    # For the WINNING strategy only: check how many of its param combos
    # also produced positive Sharpe on training data.
    winning_sharpes = []
    for params in STRATEGY_GRIDS[best_strat]:
        try:
            pos = STRATEGY_FNS[best_strat](train_price, **params)
            m   = compute_metrics(train_price, pos)
            if not np.isnan(m["Sharpe"]):
                winning_sharpes.append(m["Sharpe"])
        except:
            continue

    n_positive     = sum(1 for s in winning_sharpes if s > 0)
    n_total        = len(winning_sharpes)
    robustness_pct = (n_positive / n_total * 100) if n_total > 0 else 0

    if robustness_pct >= 75:
        robustness = "✓ High"
    elif robustness_pct >= 50:
        robustness = "~ OK"
    else:
        robustness = "⚠ Low"

    # ── Apply best config to TEST data ─────────────────────────────────────────
    pos          = STRATEGY_FNS[best_strat](test_price, **best_params)
    test_metrics = compute_metrics(test_price, pos)

    # ── B&H reference metrics ──────────────────────────────────────────────────
    log_ret    = np.log(test_price / test_price.shift(1)).dropna()
    bah_ret    = (np.exp(log_ret.cumsum()).iloc[-1] - 1) * 100 if len(log_ret) > 0 else np.nan
    bah_ann    = np.exp(log_ret.mean() * PERIODS_PER_YEAR) - 1
    bah_vol    = log_ret.std() * np.sqrt(PERIODS_PER_YEAR)
    bah_sharpe = bah_ann / bah_vol if bah_vol != 0 else np.nan

    # ── B&H Max Drawdown ───────────────────────────────────────────────────────
    # How bad did it get if you just held the stock the whole time?
    # DD Saved % = Strategy Max DD − B&H Max DD
    #   Positive = strategy protected you (e.g. strategy -15%, B&H -30% → saved 15%)
    #   Negative = strategy actually had a worse drawdown than just holding
    bah_cum = np.exp(log_ret.cumsum())
    bah_mdd = ((bah_cum - bah_cum.cummax()) / bah_cum.cummax()).min() * 100 if len(bah_cum) > 0 else np.nan
    dd_saved = round(test_metrics["Max DD %"] - float(bah_mdd), 2) if not np.isnan(bah_mdd) else np.nan

    best_configs[asset] = {"strategy": best_strat, "params": best_params,
                           "train_sharpe": round(best_sharpe, 3)}
    test_dfs[asset] = test_metrics["_df"]

    screener_results.append({
        "Asset":              asset,
        "Best Strategy":      best_strat,
        "Best Params":        str(best_params),
        "Train Sharpe":       round(best_sharpe, 3),
        "Robustness":         robustness,
        "Robust %":           round(robustness_pct, 0),
        "OUT Sharpe":         test_metrics["Sharpe"],
        "OUT Strat Ret %":    test_metrics["Total Ret %"],
        "OUT B&H Ret %":      round(float(bah_ret), 2),
        "OUT B&H Sharpe":     round(float(bah_sharpe), 3),
        "OUT Strat Max DD %": test_metrics["Max DD %"],
        "OUT B&H Max DD %":   round(float(bah_mdd), 2),
        "DD Saved %":         dd_saved,
        "Beats B&H":          test_metrics["Total Ret %"] > bah_ret,
        "Run Date":           dt.datetime.now().strftime("%Y-%m-%d"),
    })

    beats = "✓" if screener_results[-1]["Beats B&H"] else "✗"
    dd_saved_str = f"{dd_saved:+.1f}%" if not np.isnan(dd_saved) else "N/A"
    print(f"{best_strat} | Robust: {robustness} ({robustness_pct:.0f}%) | "
          f"OUT Sharpe: {test_metrics['Sharpe']:.2f} | "
          f"Strat DD: {test_metrics['Max DD %']:.1f}% | "
          f"B&H DD: {bah_mdd:.1f}% | DD Saved: {dd_saved_str} | "
          f"Ret: {test_metrics['Total Ret %']:.0f}% vs B&H: {bah_ret:.0f}% {beats}")


# ── Build results dataframe ────────────────────────────────────────────────────
results_df = pd.DataFrame(screener_results).set_index("Asset")

print("\n" + "=" * 80)
print("  SCREENER COMPLETE")
print("=" * 80)
print(results_df[["Best Strategy", "Robustness", "OUT Sharpe",
                   "OUT Strat Ret %", "OUT Strat Max DD %",
                   "OUT B&H Max DD %", "DD Saved %", "Beats B&H"]].to_string())

beats_count = results_df["Beats B&H"].sum()
print(f"\nAssets beating B&H: {beats_count} / {len(results_df)}")
print(f"\nDD Saved leaders (top 5 — most drawdown protection):")
top_dd = results_df.nlargest(5, "DD Saved %")[["Best Strategy", "OUT Strat Max DD %", "OUT B&H Max DD %", "DD Saved %"]]
print(top_dd.to_string())
print(f"\nRobustness breakdown:")
print(results_df["Robustness"].value_counts().to_string())
print(f"\nStrategy distribution:")
print(results_df["Best Strategy"].value_counts().to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXCEL OUTPUT — colour-coded with robustness + Max DD + DD Saved %
# ══════════════════════════════════════════════════════════════════════════════
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

today     = dt.datetime.now().strftime("%Y-%m-%d")
xlsx_path = os.path.join(OUTPUT_DIR, f"Goofy_Screener_{today}.xlsx")

export_df = results_df.drop(columns=["Best Params", "Run Date"], errors="ignore").reset_index()
export_df.to_excel(xlsx_path, index=False, sheet_name="Screener Results")

wb = load_workbook(xlsx_path)
ws = wb["Screener Results"]

STRAT_FILLS = {
    "MA Crossover":    PatternFill("solid", fgColor="AED6F1"),
    "RSI":             PatternFill("solid", fgColor="FAD7A0"),
    "Bollinger Bands": PatternFill("solid", fgColor="A9DFBF"),
    "MACD":            PatternFill("solid", fgColor="D7BDE2"),
    "Mean Reversion":  PatternFill("solid", fgColor="F1948A"),
}
ROBUST_FILLS = {
    "✓ High": PatternFill("solid", fgColor="D5F5E3"),
    "~ OK":   PatternFill("solid", fgColor="FEF9E7"),
    "⚠ Low":  PatternFill("solid", fgColor="FADBD8"),
}
GREEN_FILL  = PatternFill("solid", fgColor="D5F5E3")
RED_FILL    = PatternFill("solid", fgColor="FADBD8")
YELLOW_FILL = PatternFill("solid", fgColor="FEF9E7")
HDR_FILL    = PatternFill("solid", fgColor="2C3E50")
HDR_FONT    = Font(bold=True, color="FFFFFF")
THIN        = Side(style="thin", color="CCCCCC")
BORDER      = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

headers       = [c.value for c in ws[1]]
strat_col     = headers.index("Best Strategy")      + 1 if "Best Strategy"      in headers else None
robust_col    = headers.index("Robustness")          + 1 if "Robustness"          in headers else None
beats_col     = headers.index("Beats B&H")           + 1 if "Beats B&H"           in headers else None
sharpe_col    = headers.index("OUT Sharpe")          + 1 if "OUT Sharpe"          in headers else None
strat_mdd_col = headers.index("OUT Strat Max DD %")  + 1 if "OUT Strat Max DD %"  in headers else None
bah_mdd_col   = headers.index("OUT B&H Max DD %")    + 1 if "OUT B&H Max DD %"    in headers else None
dd_saved_col  = headers.index("DD Saved %")          + 1 if "DD Saved %"          in headers else None

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = BORDER
ws.row_dimensions[1].height = 30

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    strat_val  = row[strat_col  - 1].value if strat_col  else None
    robust_val = row[robust_col - 1].value if robust_col else None
    beats_val  = row[beats_col  - 1].value if beats_col  else None

    for cell in row:
        cell.border    = BORDER
        cell.alignment = Alignment(horizontal="center", vertical="center")

        if strat_col and cell.column == strat_col:
            cell.fill = STRAT_FILLS.get(strat_val, PatternFill())
            cell.font = Font(bold=True)

        if robust_col and cell.column == robust_col:
            cell.fill = ROBUST_FILLS.get(robust_val, PatternFill())
            cell.font = Font(bold=True)

        if beats_col and cell.column == beats_col:
            cell.fill = GREEN_FILL if beats_val else RED_FILL

        if sharpe_col and cell.column == sharpe_col:
            try:
                v = float(cell.value)
                if v >= 0.7:         cell.fill = GREEN_FILL
                elif 0.4 <= v < 0.7: cell.fill = YELLOW_FILL
                elif v < 0:          cell.fill = RED_FILL
            except: pass

        # Strategy Max DD — negative number, closer to 0 = better
        if strat_mdd_col and cell.column == strat_mdd_col:
            try:
                v = float(cell.value)
                if v >= -15:          cell.fill = GREEN_FILL    # small drawdown ✓
                elif -30 <= v < -15:  cell.fill = YELLOW_FILL
                elif v < -30:         cell.fill = RED_FILL      # large drawdown ✗
            except: pass

        # B&H Max DD — same colour scale (just for reference)
        if bah_mdd_col and cell.column == bah_mdd_col:
            try:
                v = float(cell.value)
                if v >= -15:          cell.fill = GREEN_FILL
                elif -30 <= v < -15:  cell.fill = YELLOW_FILL
                elif v < -30:         cell.fill = RED_FILL
            except: pass

        # DD Saved % — POSITIVE = strategy protected you from drawdown
        # The higher the better! Green = saved >10%, Red = strategy was worse
        if dd_saved_col and cell.column == dd_saved_col:
            try:
                v = float(cell.value)
                if v >= 10:          cell.fill = GREEN_FILL    # saved ≥10% drawdown
                elif 0 <= v < 10:    cell.fill = YELLOW_FILL   # mild protection
                elif v < 0:          cell.fill = RED_FILL      # strategy had worse DD than holding!
            except: pass

for col in ws.columns:
    w = max((len(str(c.value or "")) for c in col), default=0)
    ws.column_dimensions[get_column_letter(col[0].column)].width = min(w + 4, 30)
ws.freeze_panes = "A2"

# ── Summary sheet ──────────────────────────────────────────────────────────────
ws2 = wb.create_sheet("Summary")
ws2.append(["Goofy Screener v2 — Daily Summary", ""])
ws2.append(["Run Date", today])
ws2.append(["Universe", UNIVERSE])
ws2.append(["Assets Screened", len(results_df)])
ws2.append(["Assets Beating B&H", int(results_df["Beats B&H"].sum())])
ws2.append(["", ""])
ws2.append(["Robustness Breakdown", ""])
for rob, count in results_df["Robustness"].value_counts().items():
    ws2.append([rob, count])
ws2.append(["", ""])
ws2.append(["Strategy Distribution", ""])
for strat, count in results_df["Best Strategy"].value_counts().items():
    ws2.append([strat, count])
ws2.append(["", ""])
ws2.append(["Top 5 DD Protectors (highest DD Saved %)", ""])
ws2.append(["Asset", "Strategy", "Robustness", "Strat Max DD %", "B&H Max DD %", "DD Saved %"])
top_dd = results_df.nlargest(5, "DD Saved %")[["Best Strategy", "Robustness",
                                                "OUT Strat Max DD %", "OUT B&H Max DD %", "DD Saved %"]]
for asset, row in top_dd.iterrows():
    ws2.append([asset, row["Best Strategy"], row["Robustness"],
                row["OUT Strat Max DD %"], row["OUT B&H Max DD %"], row["DD Saved %"]])
ws2.append(["", ""])
ws2.append(["Top 5 by OUT Sharpe (High Robustness only)", ""])
high_robust = results_df[results_df["Robustness"] == "✓ High"]
top5 = high_robust.nlargest(5, "OUT Sharpe")[["Best Strategy", "Robustness",
                                               "OUT Sharpe", "OUT Strat Ret %",
                                               "OUT Strat Max DD %", "OUT B&H Max DD %", "DD Saved %"]]
ws2.append(["Asset", "Strategy", "Robustness", "OUT Sharpe", "OUT Strat Ret %",
            "Strat Max DD %", "B&H Max DD %", "DD Saved %"])
for asset, row in top5.iterrows():
    ws2.append([asset, row["Best Strategy"], row["Robustness"],
                row["OUT Sharpe"], row["OUT Strat Ret %"],
                row["OUT Strat Max DD %"], row["OUT B&H Max DD %"], row["DD Saved %"]])
ws2["A1"].font = Font(bold=True, size=13)

wb.save(xlsx_path)
print(f"Excel saved: {xlsx_path}")
print(f"\nTop 5 HIGH-ROBUSTNESS assets by OUT Sharpe:")
print(top5.to_string())
print(f"\nTop 5 DD Protectors (DD Saved %):")
print(top_dd.to_string())


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
#  CHARTS
#  1. Strategy distribution pie + Sharpe bar
#  2. Risk-Return scatter: Sharpe vs Max DD — the key "is it worth it?" chart
#  3. Equity curves for top 6 HIGH-ROBUSTNESS assets
# ══════════════════════════════════════════════════════════════════════════════

today = dt.datetime.now().strftime("%Y-%m-%d")

# ── Chart 1: Pie + Sharpe bar ──────────────────────────────────────────────────
strat_counts = results_df["Best Strategy"].value_counts()
colors_pie   = [STRATEGY_COLORS.get(s, "grey") for s in strat_counts.index]

fig1, axes1 = plt.subplots(1, 2, figsize=(16, 6))

axes1[0].pie(strat_counts.values, labels=strat_counts.index,
             colors=colors_pie, autopct="%1.0f%%", startangle=90)
axes1[0].set_title(f"Best Strategy Distribution\n({UNIVERSE} Universe)", fontsize=12)

top_n  = results_df.nlargest(min(15, len(results_df)), "OUT Sharpe")
colors = [STRATEGY_COLORS.get(s, "grey") for s in top_n["Best Strategy"]]
bars   = axes1[1].bar(range(len(top_n)), top_n["OUT Sharpe"], color=colors, alpha=0.85)

for i, (bar, (asset, row)) in enumerate(zip(bars, top_n.iterrows())):
    rob_marker = "✓" if row["Robustness"] == "✓ High" else ("~" if row["Robustness"] == "~ OK" else "⚠")
    axes1[1].text(bar.get_x() + bar.get_width()/2,
                  bar.get_height() + 0.01,
                  f"{asset}\n{row['OUT Sharpe']:.2f} {rob_marker}",
                  ha="center", va="bottom", fontsize=7)

legend_els = [mpatches.Patch(facecolor=c, label=s) for s, c in STRATEGY_COLORS.items()]
axes1[1].legend(handles=legend_els, fontsize=7, loc="upper right")
axes1[1].set_xticks([])
axes1[1].set_ylabel("OUT Sharpe Ratio")
axes1[1].set_title(f"Top Assets by OUT Sharpe\n(✓=High Robust  ~=OK  ⚠=Low)", fontsize=12)
axes1[1].axhline(y=0, color="black", linewidth=0.8)
axes1[1].grid(True, alpha=0.3, axis="y")

plt.suptitle(f"Goofy Screener v2 — {UNIVERSE} Universe | {today}", fontsize=13)
plt.tight_layout()
chart1_path = os.path.join(OUTPUT_DIR, f"screener_summary_{today}.png")
plt.savefig(chart1_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {chart1_path}")


# ── Chart 2: Risk-Return scatter (Sharpe vs Max DD) ───────────────────────────
# This is the most important chart for deciding whether a strategy is worth using.
# X axis = OUT Sharpe (higher = better risk-adjusted return)
# Y axis = OUT Max DD % (less negative = smaller potential loss)
# Bubble size = OUT Strat Ret % (bigger bubble = higher absolute return)
# Colour = strategy
# Robustness shown by marker shape: circle = High, triangle = OK, X = Low
#
# IDEAL ZONE: top-right = high Sharpe + small drawdown = best strategies
# AVOID ZONE: bottom-left = low Sharpe + large drawdown = worst of both worlds

fig2, ax2 = plt.subplots(figsize=(14, 8))

valid = results_df.dropna(subset=["OUT Sharpe", "OUT Max DD %"])

for _, row in valid.iterrows():
    asset  = row.name
    strat  = row["Best Strategy"]
    rob    = row["Robustness"]
    color  = STRATEGY_COLORS.get(strat, "grey")
    size   = max(abs(row["OUT Strat Ret %"]) * 3, 30)
    marker = "o" if rob == "✓ High" else ("^" if rob == "~ OK" else "X")
    alpha  = 0.9 if rob == "✓ High" else 0.6

    ax2.scatter(row["OUT Sharpe"], row["OUT Max DD %"],
                s=size, c=color, marker=marker, alpha=alpha,
                edgecolors="white", linewidths=0.5, zorder=3)
    ax2.annotate(asset,
                 xy=(row["OUT Sharpe"], row["OUT Max DD %"]),
                 xytext=(5, 3), textcoords="offset points",
                 fontsize=7, color="black")

# Quadrant lines
ax2.axvline(x=0.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.5, label="_nolegend_")
ax2.axhline(y=-20, color="grey", linestyle="--", linewidth=0.8, alpha=0.5, label="_nolegend_")

# Quadrant labels
ax2.text(0.97, 0.97, "✓ IDEAL\nHigh Sharpe\nSmall Drawdown",
         transform=ax2.transAxes, fontsize=8, va="top", ha="right",
         color="green", alpha=0.6)
ax2.text(0.03, 0.03, "✗ AVOID\nLow Sharpe\nLarge Drawdown",
         transform=ax2.transAxes, fontsize=8, va="bottom", ha="left",
         color="red", alpha=0.6)

# Strategy legend
strat_legend = [mpatches.Patch(facecolor=c, label=s) for s, c in STRATEGY_COLORS.items()]
# Robustness legend
from matplotlib.lines import Line2D
rob_legend = [
    Line2D([0],[0], marker="o", color="w", markerfacecolor="grey", markersize=8, label="✓ High Robustness"),
    Line2D([0],[0], marker="^", color="w", markerfacecolor="grey", markersize=8, label="~ OK Robustness"),
    Line2D([0],[0], marker="X", color="w", markerfacecolor="grey", markersize=8, label="⚠ Low Robustness"),
]
l1 = ax2.legend(handles=strat_legend, loc="upper left", fontsize=8, title="Strategy")
ax2.add_artist(l1)
ax2.legend(handles=rob_legend, loc="lower right", fontsize=8, title="Robustness")

ax2.set_xlabel("OUT Sharpe Ratio (higher = better risk-adjusted return)", fontsize=10)
ax2.set_ylabel("OUT Max Drawdown % (less negative = smaller potential loss)", fontsize=10)
ax2.set_title(f"Risk-Return Map — Sharpe vs Max Drawdown\n"
              f"Bubble size = strategy return % | {UNIVERSE} Universe | {today}", fontsize=12)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
chart2_path = os.path.join(OUTPUT_DIR, f"screener_risk_return_{today}.png")
plt.savefig(chart2_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {chart2_path}")


# ── Chart 3: Equity curves — top 6 HIGH ROBUSTNESS assets ─────────────────────
high_robust_assets = (results_df[results_df["Robustness"] == "✓ High"]
                      .nlargest(min(6, len(results_df)), "OUT Sharpe")
                      .index.tolist())

# Fall back to top Sharpe overall if not enough High Robust assets
if len(high_robust_assets) < 6:
    extra = (results_df[~results_df.index.isin(high_robust_assets)]
             .nlargest(6 - len(high_robust_assets), "OUT Sharpe")
             .index.tolist())
    high_robust_assets.extend(extra)

fig3, axes3 = plt.subplots(2, 3, figsize=(18, 10))
axes3 = axes3.flatten()

for i, asset in enumerate(high_robust_assets):
    ax    = axes3[i]
    df    = test_dfs[asset]
    strat = results_df.loc[asset, "Best Strategy"]
    rob   = results_df.loc[asset, "Robustness"]
    color = STRATEGY_COLORS.get(strat, "orange")
    out_s = results_df.loc[asset, "OUT Sharpe"]
    mdd   = results_df.loc[asset, "OUT Max DD %"]

    ax.plot(df.index, df["cum_market"],   color="steelblue",
            linewidth=1.5, linestyle="--", label="Buy & Hold")
    ax.plot(df.index, df["cum_strategy"], color=color,
            linewidth=1.5, label=strat)
    ax.axhline(y=1, color="gray", linestyle=":", linewidth=0.8)
    ax.set_title(f"{asset}  |  {strat}  [{rob}]\n"
                 f"Sharpe: {out_s:.2f}  |  Max DD: {mdd:.1f}%", fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

for j in range(i + 1, len(axes3)):
    axes3[j].set_visible(False)

plt.suptitle(f"Top Assets — Equity Curves (Out-of-Sample 2021–2026)\n"
             f"{UNIVERSE} Universe — prioritising High Robustness | {today}",
             fontsize=12, y=1.01)
plt.tight_layout()
chart3_path = os.path.join(OUTPUT_DIR, f"screener_equity_{today}.png")
plt.savefig(chart3_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {chart3_path}")

print(f"\nAll outputs saved to: {OUTPUT_DIR}/")
